In [ ]:
# Defining an enviroment for the agent (logistics center)
import numpy as np
import random as rd
import copy as cp

# The logistics center is created according to the docks number and SKU numbers
# SKU number = the different number of products avaible in the warehouse
# docker number = the different number of docks in the warehouse
class logistics_center:

    def __init__(self, num_docks = 3, num_SKU = 3):
        self.num_docks = num_docks
        self.num_SKU = num_SKU
        self.my_table = Qtable(lr = 0.01, gamma = 0.2)

    # the warehouse is a matrix, that contains zeros as "empty space", ones which are the docks, and twos, which are the the SKU positions
    def create_warehouse(self):

        warehouse = np.zeros((self.num_docks, (self.num_docks + 2)))

        for i in range(self.num_docks):
            warehouse[i,0] = 1

        j = 0 

        while j < self.num_SKU:
            a , b = rd.randint(0, (self.num_docks -1)), rd.randint(1, (self.num_docks + 1))
            if warehouse[a,b] == 0:
                warehouse[a, b] = 2
                j += 1

        return warehouse

    # This function is responsible for reseting the enviroment for a new episode to start
    def reset(self):
        self.warehouse = self.create_warehouse()
        self.state = State(self.warehouse)          
        self.state.create_state(self.warehouse) 
        self.agent = Agent(self.warehouse, self.state)


    #here is where we define the episode, where currently the agent does random moves
    def episode(self):
        self.reset()
        p_state = cp.deepcopy(self.state)
        actions_list =[self.agent.action_down, self.agent.action_up, self.agent.action_left, self.agent.action_right]
        
        while True:
            b = 0
            done = False
            a = self.my_table.choose_action(state = self.state)
            actions_list[a]()
            reward = self.reward(self.agent)

            if self.state.has_SKU[1] >= 1 and self.state.deadline[1] >= 0:
                b += 1
                print(f"Success!{b} deliveries")
                self.my_table.update_q_table(state = p_state, action = a, reward = reward, next_state = self.state, done = done)
                p_state = cp.deepcopy(self.state)
                self.state.SKU_requisition = rd.choice(np.argwhere(self.warehouse == 2))
                self.state.delivery_dock = rd.choice(np.argwhere(self.warehouse == 1))
                self.state.has_SKU = np.array([0, 0])
                p_state = cp.deepcopy(self.state)

            if self.state.deadline[0] >= 10:
                print("Fail")
                done = True
                self.my_table.update_q_table(state = p_state, action = a, reward = reward, next_state = self.state, done = done)
                break

            self.my_table.update_q_table(state = p_state, action = a, reward = reward, next_state = self.state, done = done)
            p_state = cp.deepcopy(self.state)

            print("Position:", self.agent.position)
            print("Reward:", reward)



    #the reward definition, while the agent is moving around reward = -1, the moment he reaches a SKU reward = 5 andwhen he reaches a dock reward = 10
    def reward(self, agent):
        reward = 0

        reward -= 1

        if agent.has_SKU[0] == 1 and agent.just_picked_SKU:
            reward +=5

        if agent.has_SKU[1] >= 1 and agent.just_delivered_SKU:
            reward += 10

        if agent.deadline[1] == 0 and agent.has_SKU[1] == 0:
            reward -= 10 

        return reward


In [3]:
#defining the state, which is going to have the SKU requisition, the agent position, where does he need to go, deadline and if he is carrying an SKU number
#the state is a matrix, that gives to the agent the following informations:
#where's he at, where's the dock he is supposed to go, where's the SKU he is supposed to go, a deadline the amount of moves he did/can do and 

class State(logistics_center):

    def __init__(self, created_warehouse):
        self.created_warehouse = created_warehouse

    def create_state(self, created_warehouse):

        state_list = []
        
        empty_positions = np.argwhere(created_warehouse == 0)
        self.position = np.array(rd.choice(empty_positions))
        self.position = [int(x)for x in self.position]
        state_list.append(self.position)
        

        dock_positions = np.argwhere(created_warehouse == 1)
        self.delivery_dock = np.array(rd.choice(dock_positions))
        self.delivery_dock = [int(x)for x in self.delivery_dock]
        state_list.append(self.delivery_dock)
        

        sku_positions = np.argwhere(created_warehouse == 2)
        self.SKU_requisition = np.array(rd.choice(sku_positions))
        self.SKU_requisition = [int(x)for x in self.SKU_requisition]
        state_list.append(self.SKU_requisition)



        self.deadline = np.array([0, 10]) # the agent can do 10 "moves" to get to accomplish his mission, deadline[0] = amount of moves done, deadline[1] = moves left
        state_list.append(self.deadline)

        self.has_SKU = np.array([0, 0]) # [0] = 0 , no SKU, [1] = 1, has SKU; second value just because we need a 2 column vector
        state_list.append(self.has_SKU)
    
        state = np.array(state_list)

        return state
        

In [4]:
# defining the agent, which is going to have actions and get the state and warehouse created
# the agent can move upwards, downwards and go to the left or to the right, movimentation implemented via matrix 
# we also change the state here, by moving the agent, the dealine is changed as well as the has_SKU state

class Agent():

    def __init__(self, warehouse_created, state):

        self.warehouse_created = warehouse_created
        self.state = state 
        self.just_picked_SKU = False
        self.just_delivered_SKU = False

    @property
    def position(self):
        return self.state.position

    @property
    def deadline(self):
        return self.state.deadline

    @property
    def has_SKU(self):
        return self.state.has_SKU

    def action_up(self):
        self.just_picked_SKU = False
        self.just_delivered_SKU = False
        self.matrix_up = np.array([-1, 0])

        if self.position[0] - 1 >= 0:
            new_position = np.array(self.position) + self.matrix_up
            self.state.position[:] = [int(x) for x in new_position]   
            self.state.deadline += np.array([1, -1])
            if self.warehouse_created[self.position[0], self.position[1]] == 2 and self.has_SKU[0] == 0:
                self.state.has_SKU += np.array([1, 0])
                self.just_picked_SKU = True

            if self.warehouse_created[self.position[0], self.position[1]] == 1 and self.has_SKU[0] == 1:
                self.state.has_SKU += np.array([-1, 0])
                self.just_delivered_SKU = True

    def action_down(self):
        self.just_picked_SKU = False
        self.just_delivered_SKU = False
        self.matrix_down = np.array([1, 0])

        if self.position[0] + 1 < len(self.warehouse_created):
            new_position = np.array(self.position) + self.matrix_down
            self.state.position[:] = [int(x) for x in new_position]
            self.state.deadline += np.array([1, -1])
            if self.warehouse_created[self.position[0], self.position[1]] == 2 and self.has_SKU[0] == 0:
                self.state.has_SKU += np.array([1, 0])
                self.just_picked_SKU = True

            if self.warehouse_created[self.position[0], self.position[1]] == 1 and self.has_SKU[0] == 1:
                self.state.has_SKU += np.array([-1, 0])
                self.just_delivered_SKU = True

    def action_left(self):
        self.just_picked_SKU = False
        self.just_delivered_SKU = False
        self.matrix_left = np.array([0, -1])

        if self.position[1] - 1 >= 0:
            new_position = np.array(self.position) + self.matrix_left
            self.state.position[:] = [int(x) for x in new_position]
            self.state.deadline += np.array([1, -1])
            if self.warehouse_created[self.position[0], self.position[1]] == 2 and self.has_SKU[0] == 0:
                self.state.has_SKU += np.array([1, 0])
                self.just_picked_SKU = True

            if self.warehouse_created[self.position[0], self.position[1]] == 1 and self.has_SKU[0] == 1:
                self.state.has_SKU += np.array([-1, 0])
                self.just_delivered_SKU = True

    def action_right(self):
        self.just_picked_SKU = False
        self.just_delivered_SKU = False
        self.matrix_right = np.array([0, 1])

        if self.position[1] + 1 < len(self.warehouse_created[0]):
            new_position = np.array(self.position) + self.matrix_right
            self.state.position[:] = [int(x) for x in new_position]
            self.state.deadline += np.array([1, -1])
            if self.warehouse_created[self.position[0], self.position[1]] == 2 and self.has_SKU[0] == 0:
                self.state.has_SKU += np.array([1, 0])
                self.just_picked_SKU = True

            if self.warehouse_created[self.position[0], self.position[1]] == 1 and self.has_SKU[0] == 1:
                self.state.has_SKU += np.array([-1, 0])
                self.just_delivered_SKU = True

In [5]:
#the first model we will try it's gonna be Q-Learning

class Qtable():

    def __init__(self, lr, gamma):
       
        self.lr = lr
        self.gamma = gamma

    def create_q_table(self):

        num_states = (logistics_center.num_docks*(logistics_center.num_docks+2)*logistics_center.num_docks*logistics_center.num_SKU*11*11*2)
        self.q_table = np.zeros((num_states,4))
        return self.q_table


    def state_to_int(self, state):
        position = state[0][0] * 5 + state[0][1]
        dock = state[1][0] * 3 + state[1][1]
        sku = state[2][0] * 3 + state[2][1]
        deadline = state[3][0] * 11 + state[3][1]
        has_sku = state[4][0]

        state_number = (
            position * (3 * 3 * 121 * 2)
            + dock * (3 * 121 * 2)
            + sku * (121 * 2)
            + deadline * 2
            + has_sku
        )
        return state_number

    def choose_action(self,state):

        current_state = self.state_to_int(state)
        action = np.argmax(self.q_table[current_state])
        return action


    def update_q_table(self, state, action, reward, next_state, done):

        current_state = self.state_to_int(state)
        next_state = self.state_to_int(next_state)

        current_q = self.q_table[current_state, action]

        if done:
            target = reward
        else:
            max_next_q = np.max(self.q_table[next_state])
            target = reward + self.gamma * max_next_q

        self.q_table[current_state, action] = (
            current_q + self.learning_rate * (target - current_q)
        )


In [7]:
env = logistics_center()

env.reset()

actions = [
    env.agent.action_down,
    env.agent.action_up,
    env.agent.action_left,
    env.agent.action_right
]

for step in range(100):

    action = rd.choice(actions)
    action()

    reward = env.reward(env.agent)

    print(f"\nStep: {step + 1}")
    print("Position:", env.agent.position)
    print("SKU requisition:", env.state.SKU_requisition)
    print("Delivery dock:", env.state.delivery_dock)
    print("Deadline:", env.state.deadline)
    print("Has SKU:", env.state.has_SKU)
    print("Reward:", reward)

    # Entrega realizada
    if env.state.has_SKU[1] >= 1:
        print(">>> SKU delivered!")

        env.state.SKU_requisition = rd.choice(
            np.argwhere(env.warehouse == 2)
        )

        env.state.delivery_dock = rd.choice(
            np.argwhere(env.warehouse == 1)
        )

        env.state.has_SKU = np.array([0, 0])

        print(">>> New SKU:", env.state.SKU_requisition)
        print(">>> New dock:", env.state.delivery_dock)

    # Deadline atingido
    if env.state.deadline[0] >= 10:
        print(">>> Deadline reached. Episode finished.")
        break


Step: 1
Position: [1, 1]
SKU requisition: [0, 3]
Delivery dock: [1, 0]
Deadline: [1 9]
Has SKU: [0 0]
Reward: -1

Step: 2
Position: [1, 2]
SKU requisition: [0, 3]
Delivery dock: [1, 0]
Deadline: [2 8]
Has SKU: [0 0]
Reward: -1

Step: 3
Position: [1, 1]
SKU requisition: [0, 3]
Delivery dock: [1, 0]
Deadline: [3 7]
Has SKU: [0 0]
Reward: -1

Step: 4
Position: [1, 0]
SKU requisition: [0, 3]
Delivery dock: [1, 0]
Deadline: [4 6]
Has SKU: [0 0]
Reward: -1

Step: 5
Position: [0, 0]
SKU requisition: [0, 3]
Delivery dock: [1, 0]
Deadline: [5 5]
Has SKU: [0 0]
Reward: -1

Step: 6
Position: [0, 0]
SKU requisition: [0, 3]
Delivery dock: [1, 0]
Deadline: [5 5]
Has SKU: [0 0]
Reward: -1

Step: 7
Position: [1, 0]
SKU requisition: [0, 3]
Delivery dock: [1, 0]
Deadline: [6 4]
Has SKU: [0 0]
Reward: -1

Step: 8
Position: [0, 0]
SKU requisition: [0, 3]
Delivery dock: [1, 0]
Deadline: [7 3]
Has SKU: [0 0]
Reward: -1

Step: 9
Position: [0, 0]
SKU requisition: [0, 3]
Delivery dock: [1, 0]
Deadline: [7 3]


Coletar múltiplos pedidos de uma vez, uma flag de prioridade para várias rotas, armazém mais restrito com regras de movimentações (ele precisa passar pelos corredores) cada elemento da matriz é 1m (ex) e estima-se o tempo por uma "velocidade do agente". Movimentação + roteirização/sequenciamento de tarefas